# Ciringoni M4 training native tiga puluh epoch

Latih pasangan native 640 dan 1280 dari bobot awal yang identik. Dataset terpasang hanya berisi train dan validation.


In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys
WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = "https://github.com/Jordyee/ciringoni-data-mining.git"
SOURCE = "ee8c14bd132a07db3dae4bb15de0d8a22f517d60"
PROJECT = Path("/tmp/ciringoni-source/Data_Mining_Ciringoni_M1")
SCRIPTS = PROJECT / "hasil_m2"

def execute(*args):
    subprocess.run([str(arg) for arg in args], cwd=PROJECT, check=True)

def input_file(name):
    paths = list(INPUT.rglob(name))
    assert len(paths) == 1, f"Expected one mounted {name}, found {len(paths)}"
    return paths[0]

subprocess.run(["git", "clone", "--quiet", REPO, str(Path("/tmp/ciringoni-source"))], check=True)
execute("git", "checkout", "--quiet", SOURCE)


## Input dan lingkungan

Validasi identitas sumber, direktori, dan bobot awal sebelum latihan.


In [ ]:
execute(sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.210")
from tarfile import open as tar_open
from zipfile import ZipFile

NATIVE = Path("/tmp/ciringoni-native")
NATIVE.mkdir()
archive_paths = list(INPUT.rglob("native_train_val.tar")) + list(INPUT.rglob("native_train_val.zip"))
if archive_paths:
    assert len(archive_paths) == 1
    archive = archive_paths[0]
    if archive.suffix == ".zip":
        with ZipFile(archive) as packed:
            packed.extractall(NATIVE)
    else:
        with tar_open(archive) as packed:
            packed.extractall(NATIVE, filter="data")
    if not (NATIVE / "640").is_dir():
        roots = [path.parent for path in NATIVE.rglob("640") if (path / "images").is_dir()]
        assert len(roots) == 1
        NATIVE = roots[0]
else:
    roots = [path.parent for path in INPUT.rglob("640")
             if (path / "images/train").is_dir() and (path.parent / "1280/images/train").is_dir()]
    assert len(roots) == 1, f"Expected one native input root, found {len(roots)}"
    for resolution in ("640", "1280"):
        shutil.copytree(roots[0] / resolution, NATIVE / resolution)
if not (NATIVE / "audit_pasangan_e2.json").is_file():
    shutil.copyfile(input_file("audit_pasangan_e2.json"), NATIVE / "audit_pasangan_e2.json")
for size in (640, 1280):
    root = NATIVE / str(size)
    assert root.is_dir() and not (root / "images/test").exists()
    (root / "data.yaml").write_text(
        json.dumps({"path": str(root), "train": "images/train", "val": "images/val", "names": ["infected"]})
    )
for target, name in [(PROJECT / "data", "yolo26n.pt"), (PROJECT / "hasil_m1", "coco_val_gt.json")]:
    target.mkdir(exist_ok=True)
    source = NATIVE / name
    shutil.copyfile(source if source.is_file() else input_file(name), target / name)
with (PROJECT / "data/yolo26n.pt").open("rb") as stream:
    assert hashlib.file_digest(stream, "sha256").hexdigest() == "9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0652110fb03a8ad4fef"


## Training dan bukti run

Kedua ukuran memakai 30 epoch, batch 4, seed 42, dan AdamW. Periksa jumlah epoch sebelum menyimpan manifest. Jika tersedia dua GPU T4, satu proses menangani setiap resolusi dan memakai direktori kerja terpisah.


In [ ]:
import os, re, time
import torch, ultralytics

sys.path.insert(0, str(SCRIPTS))
from train_e2_native import verify_pair
EPOCHS = 30
BATCH = 4
TRAINED = WORK / "m4-train"
PAIR = [(640, "E2_control_native_640"), (1280, "E2_native_1280")]
visible = os.environ.get("CUDA_VISIBLE_DEVICES")
devices = visible.split(",") if visible else [str(index) for index in range(torch.cuda.device_count())]
assert torch.cuda.is_available()
gpu_names = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
verify_pair(NATIVE)
if torch.cuda.device_count() >= 2:
    TRAINED.mkdir()
    worker = """
from pathlib import Path
import sys
scripts, native, weights, output, private = map(Path, sys.argv[1:6])
size, batch, epochs = map(int, sys.argv[6:9])
name = sys.argv[9]
sys.path.insert(0, str(scripts))
from train_e2_native import smoke_yaml, train
if size == 1280:
    train(weights, smoke_yaml(native, private, batch), private, "E2_smoke_1280", size, batch, 1, smoke=True)
train(weights, native / str(size) / "data.yaml", output, name, size, batch, epochs)
"""
    jobs = {}
    weights = PROJECT / "data/yolo26n.pt"
    try:
        for (size, name), device in zip(PAIR, devices[:2], strict=True):
            private = Path(f"/tmp/m4-device-{size}")
            private.mkdir()
            (private / "yolo26n.pt").symlink_to(weights)
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=device, YOLO_CONFIG_DIR=str(private / "yolo-config"),
                       OMP_NUM_THREADS="2", MKL_NUM_THREADS="2")
            log = TRAINED / f"training_{size}.log"
            command = [sys.executable, "-u", "-c", worker, SCRIPTS, NATIVE, weights, TRAINED / "runs", private,
                       size, BATCH, EPOCHS, name]
            with log.open("w") as stream:
                jobs[size] = (subprocess.Popen([str(arg) for arg in command], cwd=private, env=env,
                                                stdout=stream, stderr=subprocess.STDOUT), log)
            print(f"Started {name} on assigned GPU", flush=True)
        previous = {}
        while jobs:
            for size, (job, log) in list(jobs.items()):
                status = job.poll()
                if status not in (None, 0):
                    raise RuntimeError(f"Training {size} failed:\n" + log.read_text()[-6000:])
                with log.open("rb") as stream:
                    stream.seek(max(0, log.stat().st_size - 40000))
                    tail = stream.read().decode("utf-8", errors="replace")
                epochs = re.findall(r"^\s*(\d+)/30\s", tail, flags=re.MULTILINE)
                if epochs and epochs[-1] != previous.get(size):
                    previous[size] = epochs[-1]
                    print(f"{size} px training at epoch {epochs[-1]}/{EPOCHS}", flush=True)
                if status == 0:
                    del jobs[size]
            if jobs:
                time.sleep(20)
    finally:
        for job, _ in jobs.values():
            if job.poll() is None:
                job.terminate()
                try:
                    job.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    job.kill()
                    job.wait()
else:
    execute(sys.executable, SCRIPTS / "train_e2_native.py", NATIVE, "--batch", BATCH, "--epochs", EPOCHS,
            "--output", TRAINED)

runs = {}
for size, name in PAIR:
    directory = TRAINED / "runs" / name
    checkpoint = directory / "weights/best.pt"
    assert len((directory / "results.csv").read_text().splitlines()) == EPOCHS + 1
    with checkpoint.open("rb") as stream:
        runs[str(size)] = {"best_pt_sha256": hashlib.file_digest(stream, "sha256").hexdigest(),
                           "path": str(checkpoint.relative_to(TRAINED)), "epochs_completed": EPOCHS,
                           "actual_batch": BATCH}
metadata = {"source_commit": SOURCE, "epochs": EPOCHS, "batch": BATCH, "seed": 42,
            "device": gpu_names[0], "visible_gpu_models": gpu_names, "parallel_same_job": len(gpu_names) >= 2,
            "torch": str(torch.__version__), "ultralytics": ultralytics.__version__, "runs": runs}
(TRAINED / "training_manifest.json").write_text(json.dumps(metadata, indent=2) + "\n")
print(json.dumps(metadata, indent=2))
